# Stage B continual-pretraining baseline gate

This notebook is the required **zero-update evaluation gate** before medical continual pretraining. It restores the promoted Stage A checkpoint, verifies the disjoint Stage B data, and records medical and general validation baselines. It does not train the model.

## Before running

1. In Colab, choose **Runtime → Change runtime type → GPU**.
2. Upload `stage-b-data.tar` to `MyDrive/medical-slm/stage-b-data.tar`.
3. Keep the promoted checkpoint at `MyDrive/medical-slm-runs/stage_a/checkpoints/checkpoint_00007250`.
4. Push the current repository changes to GitHub so the clone includes Stage B support.

The Colab profile is pinned to FP16 with a gradient scaler so an interrupted run remains resumable if Colab changes the assigned GPU between T4 and A100 sessions. Baseline evaluation itself performs no backward pass.

In [ ]:
from google.colab import drive
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')

REPOSITORY_URL = 'https://github.com/moshiur00/small-language-model-for-medical-domain.git'
REPOSITORY = Path('/content/medical-slm')
DATA_ARCHIVE = Path('/content/drive/MyDrive/medical-slm/stage-b-data.tar')
DRIVE_PARENT = Path('/content/drive/MyDrive/medical-slm-runs/stage_a/checkpoints/checkpoint_00007250')
LOCAL_PARENT = Path('/content/stage_a_parent/checkpoint_00007250')
LOCAL_REPORT = Path('/content/stage_b/stage_b_baseline.json')
DRIVE_REPORT = Path('/content/drive/MyDrive/medical-slm-runs/stage_b/stage_b_baseline.json')

assert DATA_ARCHIVE.is_file(), f'Missing {DATA_ARCHIVE}'
assert (DRIVE_PARENT / 'checkpoint_manifest.json').is_file(), f'Missing or incomplete promoted Stage A checkpoint: {DRIVE_PARENT}'
print('Inputs found.')

## Download code and install the project

In [ ]:
if not (REPOSITORY / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPOSITORY)], check=True)
else:
    subprocess.run(['git', '-C', str(REPOSITORY), 'pull', '--ff-only'], check=True)

subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', str(REPOSITORY) + '[dev]'], check=True)
source_directory = str(REPOSITORY / 'src')
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)
os.chdir(REPOSITORY)
print('Repository revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## Restore the data and promoted Stage A parent

In [ ]:
subprocess.run(['tar', '-xf', str(DATA_ARCHIVE), '-C', str(REPOSITORY)], check=True)
LOCAL_PARENT.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(DRIVE_PARENT, LOCAL_PARENT, dirs_exist_ok=True)

stage_b_shards = sorted(Path('datasets/tokenized/continual_medical_stage_b/train').glob('*.bin'))
assert len(stage_b_shards) == 107, f'Expected 107 Stage B shards, found {len(stage_b_shards)}'
assert Path('datasets/tokenized/evaluation_medical/dataset_manifest.json').is_file()
assert Path('datasets/tokenized/evaluation/dataset_manifest.json').is_file()
assert Path('artifacts/tokenizer/tokenizer.json').is_file()
print('Stage B shards:', len(stage_b_shards))
print('Parent checkpoint restored:', LOCAL_PARENT)

## Verify GPU, precision, tests, and parent initialization

In [ ]:
import importlib.util
import subprocess
import sys
import torch
from pathlib import Path

repository = Path('/content/medical-slm')
if importlib.util.find_spec('medical_slm') is None:
    assert (repository / 'pyproject.toml').is_file(), (
        'Repository is not prepared. Run the Download code and install cell first.'
    )
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repository) + '[dev]'
    ], check=True)
    sys.path.insert(0, str(repository / 'src'))
    importlib.invalidate_caches()

from medical_slm.training.precision import resolve_precision

assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
gpu_name = torch.cuda.get_device_name(0)
precision = resolve_precision('fp16', 'cuda')
print({'gpu': gpu_name, 'precision': precision.name, 'uses_grad_scaler': precision.uses_grad_scaler})

In [ ]:
subprocess.run([
    'python', '-m', 'pytest',
    'tests/test_stage_b_trainer.py',
    'tests/test_training_config.py',
    'tests/test_training_checkpoint.py',
    '-q',
], check=True)

In [ ]:
subprocess.run([
    'python', 'scripts/training/train_stage_b.py',
    '--config', 'configs/training_stage_b_colab.yaml',
    '--verify-initialization-only',
], check=True)
print('Parent/model/optimizer initialization gate: PASSED')

## Run the zero-update dual baseline

This evaluates the full medical validation split and the unchanged Stage A general validation split. Depending on the assigned GPU, it can take several minutes.

In [ ]:
subprocess.run([
    'python', 'scripts/training/train_stage_b.py',
    '--config', 'configs/training_stage_b_colab.yaml',
    '--baseline-only',
    '--baseline-output', str(LOCAL_REPORT),
], check=True)

## Verify and preserve the baseline report

In [ ]:
import json
import math

report = json.loads(LOCAL_REPORT.read_text())
medical = report['medical_validation']
general = report['general_validation']

assert report['status'] == 'passed'
assert report['optimizer_updates'] == 0
assert report['consumed_tokens'] == 0
assert report['model_parameters'] == 35_463_680
assert report['parent']['checkpoint_name'] == 'checkpoint_00007250'
assert medical['tokens'] == 997_632
assert general['tokens'] == 466_432
for result in (medical, general):
    assert math.isfinite(result['loss']) and math.isfinite(result['perplexity'])

DRIVE_REPORT.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(LOCAL_REPORT, DRIVE_REPORT)
assert DRIVE_REPORT.read_bytes() == LOCAL_REPORT.read_bytes()

print('STAGE B BASELINE GATE: PASSED')
print(f"Medical: loss={medical['loss']:.6f}, perplexity={medical['perplexity']:.3f}, tokens={medical['tokens']}")
print(f"General: loss={general['loss']:.6f}, perplexity={general['perplexity']:.3f}, tokens={general['tokens']}")
print(f"General-loss ceiling during Stage B: {report['general_loss_limit']:.6f}")
print('Saved:', DRIVE_REPORT)

## Pass criteria and next action

Proceed to the short Stage B development run only when the final cell prints `STAGE B BASELINE GATE: PASSED`. The two measured losses become immutable run baselines. During continual pretraining, a checkpoint is eligible for promotion only if medical validation improves and general validation loss remains no more than 5% above this baseline.

# One-batch alignment diagnostic

This isolated ten-update run repeatedly trains on one real packed batch. It does not create a resumable checkpoint and cannot contaminate development or full-run state.

In [ ]:
import yaml

OVERFIT_OUTPUT = Path('/content/stage_b_overfit')
OVERFIT_CONFIG = Path('/content/training_stage_b_colab_overfit.yaml')
if OVERFIT_OUTPUT.exists():
    shutil.rmtree(OVERFIT_OUTPUT)
overfit_config = yaml.safe_load(Path('configs/training_stage_b_colab.yaml').read_text())
overfit_config.update({
    'output_directory': str(OVERFIT_OUTPUT),
    'checkpoint_backup_directory': None,
    'precision': 'fp16',
    'log_interval': 1,
    'max_updates': 10,
})
OVERFIT_CONFIG.write_text(yaml.safe_dump(overfit_config, sort_keys=False))
subprocess.run([
    'python', 'scripts/training/train_stage_b.py',
    '--config', str(OVERFIT_CONFIG),
    '--overfit-one-batch',
    '--max-updates', '10',
], check=True)

overfit_records = [
    json.loads(line)
    for line in (OVERFIT_OUTPUT / 'metrics.jsonl').read_text().splitlines()
    if line.strip() and json.loads(line)['event'] == 'overfit_one_batch'
]
assert len(overfit_records) == 10
assert all(record['metrics']['optimizer_stepped'] for record in overfit_records)
assert overfit_records[-1]['metrics']['loss'] < overfit_records[0]['metrics']['loss']
print('ONE-BATCH ALIGNMENT GATE: PASSED')
print('loss:', overfit_records[0]['metrics']['loss'], '->', overfit_records[-1]['metrics']['loss'])

# Development run: update 0 to 50

Run only after the baseline gate passes. Development checkpoints use `stage_b_dev`, completely separate from the full-run `stage_b` checkpoints.

In [ ]:
import yaml

DEV_OUTPUT = Path('/content/stage_b_dev')
DEV_CHECKPOINTS = DEV_OUTPUT / 'checkpoints'
DRIVE_DEV_CHECKPOINTS = Path('/content/drive/MyDrive/medical-slm-runs/stage_b_dev/checkpoints')
DEV_CONFIG = Path('/content/training_stage_b_colab_dev.yaml')

dev_config = yaml.safe_load(Path('configs/training_stage_b_colab.yaml').read_text())
dev_config.update({
    'output_directory': str(DEV_OUTPUT),
    'checkpoint_backup_directory': str(DRIVE_DEV_CHECKPOINTS),
    'precision': 'fp16',
    'max_updates': 100,
    'validation_interval': 50,
    'checkpoint_interval': 50,
})
DEV_CONFIG.write_text(yaml.safe_dump(dev_config, sort_keys=False))

assert not (DEV_CHECKPOINTS / 'latest.json').exists(), 'Local development run exists; use the development resume cell.'
assert not (DRIVE_DEV_CHECKPOINTS / 'latest.json').exists(), 'Drive development run exists; use the development resume cell.'
subprocess.run([
    'python', 'scripts/training/train_stage_b.py',
    '--config', str(DEV_CONFIG),
    '--max-updates', '50',
], check=True)

In [ ]:
from medical_slm.training.checkpoint import verify_checkpoint

dev_pointer = json.loads((DRIVE_DEV_CHECKPOINTS / 'latest.json').read_text())
dev_checkpoint = DRIVE_DEV_CHECKPOINTS / dev_pointer['checkpoint']
verify_checkpoint(dev_checkpoint)
dev_state = json.loads((dev_checkpoint / 'trainer_state.json').read_text())
assert dev_state['update'] == 50
assert dev_state['non_finite_events'] == 0
assert dev_state['skipped_updates'] == 0
print('Development update 50: VERIFIED')
print(dev_state)

# Development restart/resume: update 50 to 100

After a disconnect, rerun the notebook setup, restore-data, and GPU cells first. This cell restores the verified Drive checkpoint and continues at the next batch cursor.

In [ ]:
import json
import subprocess
from pathlib import Path
import yaml
from medical_slm.training.checkpoint import verify_checkpoint

DEV_OUTPUT = Path('/content/stage_b_dev')
DEV_CHECKPOINTS = DEV_OUTPUT / 'checkpoints'
DRIVE_DEV_CHECKPOINTS = Path('/content/drive/MyDrive/medical-slm-runs/stage_b_dev/checkpoints')
DEV_CONFIG = Path('/content/training_stage_b_colab_dev.yaml')
assert (DRIVE_DEV_CHECKPOINTS / 'latest.json').is_file(), 'No development checkpoint exists in Drive.'

dev_config = yaml.safe_load(Path('configs/training_stage_b_colab.yaml').read_text())
dev_config.update({
    'output_directory': str(DEV_OUTPUT),
    'checkpoint_backup_directory': str(DRIVE_DEV_CHECKPOINTS),
    'precision': 'fp16',
    'max_updates': 100,
    'validation_interval': 50,
    'checkpoint_interval': 50,
})
DEV_CONFIG.write_text(yaml.safe_dump(dev_config, sort_keys=False))
DEV_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
subprocess.run(['rsync', '-a', f'{DRIVE_DEV_CHECKPOINTS}/', f'{DEV_CHECKPOINTS}/'], check=True)

subprocess.run([
    'python', 'scripts/training/train_stage_b.py',
    '--config', str(DEV_CONFIG),
    '--resume', 'latest',
    '--max-updates', '100',
], check=True)

dev_pointer = json.loads((DRIVE_DEV_CHECKPOINTS / 'latest.json').read_text())
dev_checkpoint = DRIVE_DEV_CHECKPOINTS / dev_pointer['checkpoint']
verify_checkpoint(dev_checkpoint)
dev_state = json.loads((dev_checkpoint / 'trainer_state.json').read_text())
assert dev_state['update'] == 100
assert dev_state['non_finite_events'] == 0
assert dev_state['skipped_updates'] == 0
print('Development 50 to 100 resume: VERIFIED')

# Fresh full Stage B training — standalone

Run this cell only after the baseline, update-50, and 50→100 resume gates pass. It performs one full epoch (6,840 optimizer updates). It deliberately refuses to start if a full-run checkpoint already exists; use the resume cell in that case.

In [ ]:
from google.colab import drive
from pathlib import Path
import json
import os
import shutil
import subprocess
import torch

drive.mount('/content/drive')
REPOSITORY_URL = 'https://github.com/moshiur00/small-language-model-for-medical-domain.git'
REPOSITORY = Path('/content/medical-slm')
DATA_ARCHIVE = Path('/content/drive/MyDrive/medical-slm/stage-b-data.tar')
DRIVE_PARENT = Path('/content/drive/MyDrive/medical-slm-runs/stage_a/checkpoints/checkpoint_00007250')
LOCAL_PARENT = Path('/content/stage_a_parent/checkpoint_00007250')
LOCAL_FULL_CHECKPOINTS = Path('/content/stage_b/checkpoints')
DRIVE_FULL_CHECKPOINTS = Path('/content/drive/MyDrive/medical-slm-runs/stage_b/checkpoints')
DRIVE_BASELINE = Path('/content/drive/MyDrive/medical-slm-runs/stage_b/stage_b_baseline.json')
FULL_CONFIG = Path('configs/training_stage_b_colab.yaml')

assert DATA_ARCHIVE.is_file(), f'Missing {DATA_ARCHIVE}'
assert (DRIVE_PARENT / 'checkpoint_manifest.json').is_file(), f'Missing parent {DRIVE_PARENT}'
assert DRIVE_BASELINE.is_file(), 'Run and preserve the Stage B baseline gate first.'
baseline_gate = json.loads(DRIVE_BASELINE.read_text())
assert baseline_gate['status'] == 'passed' and baseline_gate['optimizer_updates'] == 0
assert not (LOCAL_FULL_CHECKPOINTS / 'latest.json').exists(), 'Local full run exists; use the resume cell.'
assert not (DRIVE_FULL_CHECKPOINTS / 'latest.json').exists(), 'Drive full run exists; use the resume cell.'
assert torch.cuda.is_available(), 'Select a GPU runtime before training.'

if not (REPOSITORY / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPOSITORY)], check=True)
else:
    subprocess.run(['git', '-C', str(REPOSITORY), 'pull', '--ff-only'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', str(REPOSITORY) + '[dev]'], check=True)
os.chdir(REPOSITORY)
subprocess.run(['tar', '-xf', str(DATA_ARCHIVE), '-C', str(REPOSITORY)], check=True)
LOCAL_PARENT.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(DRIVE_PARENT, LOCAL_PARENT, dirs_exist_ok=True)

print('GPU:', torch.cuda.get_device_name(0))
print('Configured precision: fp16')
subprocess.run(['python', 'scripts/training/train_stage_b.py', '--config', str(FULL_CONFIG), '--verify-initialization-only'], check=True)
subprocess.run(['python', 'scripts/training/train_stage_b.py', '--config', str(FULL_CONFIG)], check=True)

# Resume interrupted full Stage B training — standalone

Use this cell in a new GPU runtime after interruption. It downloads the current code, restores all required data and the Stage A parent, verifies the latest Stage B checkpoint, and resumes with the same FP16 policy.

In [ ]:
from google.colab import drive
from pathlib import Path
import json
import os
import shutil
import subprocess
import torch
import yaml

drive.mount('/content/drive')
REPOSITORY_URL = 'https://github.com/moshiur00/small-language-model-for-medical-domain.git'
REPOSITORY = Path('/content/medical-slm')
DATA_ARCHIVE = Path('/content/drive/MyDrive/medical-slm/stage-b-data.tar')
DRIVE_PARENT = Path('/content/drive/MyDrive/medical-slm-runs/stage_a/checkpoints/checkpoint_00007250')
LOCAL_PARENT = Path('/content/stage_a_parent/checkpoint_00007250')
LOCAL_FULL_CHECKPOINTS = Path('/content/stage_b/checkpoints')
DRIVE_FULL_CHECKPOINTS = Path('/content/drive/MyDrive/medical-slm-runs/stage_b/checkpoints')
RESUME_CONFIG = Path('/content/training_stage_b_colab_resume.yaml')

assert DATA_ARCHIVE.is_file(), f'Missing {DATA_ARCHIVE}'
assert (DRIVE_PARENT / 'checkpoint_manifest.json').is_file(), f'Missing parent {DRIVE_PARENT}'
assert (DRIVE_FULL_CHECKPOINTS / 'latest.json').is_file(), 'No full Stage B checkpoint exists in Drive.'
assert torch.cuda.is_available(), 'Select a GPU runtime before resuming.'

if not (REPOSITORY / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPOSITORY)], check=True)
else:
    subprocess.run(['git', '-C', str(REPOSITORY), 'pull', '--ff-only'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', str(REPOSITORY) + '[dev]'], check=True)
os.chdir(REPOSITORY)
subprocess.run(['tar', '-xf', str(DATA_ARCHIVE), '-C', str(REPOSITORY)], check=True)
LOCAL_PARENT.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(DRIVE_PARENT, LOCAL_PARENT, dirs_exist_ok=True)
LOCAL_FULL_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
subprocess.run(['rsync', '-a', f'{DRIVE_FULL_CHECKPOINTS}/', f'{LOCAL_FULL_CHECKPOINTS}/'], check=True)

from medical_slm.training.checkpoint import verify_checkpoint
latest_pointer = json.loads((LOCAL_FULL_CHECKPOINTS / 'latest.json').read_text())
latest_name = latest_pointer['checkpoint']
latest_checkpoint = LOCAL_FULL_CHECKPOINTS / latest_name
verify_checkpoint(latest_checkpoint)
saved_config = json.loads((latest_checkpoint / 'config.json').read_text())
assert saved_config['training']['precision'] == 'fp16', 'Checkpoint precision is not FP16.'
RESUME_CONFIG.write_text(yaml.safe_dump(saved_config['training'], sort_keys=False))
state = json.loads((latest_checkpoint / 'trainer_state.json').read_text())
print('GPU:', torch.cuda.get_device_name(0))
print('Restored:', latest_name, 'update:', state['update'], 'tokens:', state['consumed_tokens'])

subprocess.run([
    'python', 'scripts/training/train_stage_b.py',
    '--config', str(RESUME_CONFIG),
    '--resume', 'latest',
], check=True)

## Full-run completion check

When training finishes, Drive must contain `final_stage_b.json`. Do not evaluate the sealed medical test split until validation-based checkpoint selection is complete.

In [ ]:
import json
from pathlib import Path
from medical_slm.training.checkpoint import verify_checkpoint

DRIVE_FULL_CHECKPOINTS = Path('/content/drive/MyDrive/medical-slm-runs/stage_b/checkpoints')
final_pointer_path = DRIVE_FULL_CHECKPOINTS / 'final_stage_b.json'
assert final_pointer_path.is_file(), 'Full Stage B training is not complete yet.'
final_pointer = json.loads(final_pointer_path.read_text())
final_checkpoint = DRIVE_FULL_CHECKPOINTS / final_pointer['checkpoint']
verify_checkpoint(final_checkpoint)
final_state = json.loads((final_checkpoint / 'trainer_state.json').read_text())
assert final_state['update'] == 6840
assert final_state['epoch'] == 1
assert final_state['non_finite_events'] == 0
assert final_state['skipped_updates'] == 0
print('FULL STAGE B STATE: VERIFIED')
print(final_state)